# CHAPTER 5: PHÂN TÍCH KỸ THUẬT VÀ XÂY DỰNG BẢNG ĐIỀU KHIỂN

## Bollinger Bands

### 1. Nhập thư viện:

In [1]:
import pandas as pd
import yfinance as yf
import talib

### 2. Tải giá cổ phiếu của IBM năm 2020:

In [ ]:
df = yf.download('IBM', 
                 start='2020-01-01',
                 end='2020-12-31',
                 progress=False,
                 auto_adjust=True)

### 3. Tính toán và vẽ biểu đồ trung bình động đơn giản (SMA):

In [ ]:
df["sma_20"] = talib.SMA(df["Close"], timeperiod=20)
(
    df[["Close", "sma_20"]].plot(title="20-day Simple Moving Average (SMA)")
)

### 4. Tính toán và vẽ đồ thị Bollinger Bands:

In [ ]:
df["bb_up"], df["bb_mid"], df["bb_low"] = talib.BBANDS(df["Close"])

fig, ax = plt.subplots()

(
    df.loc[:, ["Close", "bb_up", "bb_mid", "bb_low"]]
    .plot(ax=ax, title="Bollinger Bands (BBANDS)")
)

ax.fill_between(df.index, df["bb_low"], df["bb_up"], color="lightgray", alpha=0.4)

### 5. Tính toán và vẽ đồ thị RSI

In [ ]:
df["rsi"] = talib.RSI(df["Close"])

fig, ax = plt.subplots()
df["rsi"].plot(ax=ax, title="Relative Strength Index (RSI)")
ax.hlines(y=30,
          xmin=df.index.min(),
          xmax=df.index.max(),
          colors="red")

ax.hlines(y=70,
          xmin=df.index.min(),
          xmax=df.index.max(),
          colors="red")
plt.show()

### 6. Tính toán và vẽ biểu đồ MACD:

In [ ]:
df["macd"], df["macdsignal"], df["macdhist"] = talib.MACD(df["Close"], fastperiod=26, signalperiod=9)

fig, ax = plt.subplots(2, 1, sharex=True)

(
    df[["macd", "macdsignal"]].plot(ax=ax[0], title="Moving Average Convergence Divergence (MACD)")
)

ax[1].bar(df.index, df["macdhist"].values, label="macd_hist")
ax[1].legend()

## Thư viện `ta`

### 1. Nhập thư viện

In [ ]:
from ta import add_all_ta_features

### 2. Loại bỏ các chỉ số đã tính toán trước đó và chỉ giữ lại các cột cần thiết:

In [ ]:
df = df[["Open", "High", "Low", "Close", "Volume"]].copy()

### 3. Tính toán tất cả các chỉ báo kỹ thuật có sẵn trong thư viện `ta`

In [ ]:
df = add_all_ta_features(df, open="Open", high="High", low="Low", close="Close", volume="Volume")

## Tải chỉ số RSI được tính toán cho cổ phiếu IBM từ Alpha Vantage

### 1. Nhập thư viện:

In [ ]:
from alpha_vantage.techindicators import TechIndicators

### 2. Khởi tạo lớp TechIndicators và xác thực:

In [ ]:
ta_api = TechIndicators(key="demo", output_format="pandas")

### 3. Tải xuống chỉ số RSI cho cổ phiếu IBM:

In [ ]:
rsi_df, rsi_meta = ta_api.get_rsi(symbol="IBM", time_period=14)

### 4. Vẽ biểu đồ chỉ số RSI đã tải xuống:

In [ ]:
fig, ax = plt.subplots()
rsi_df.plot(ax=ax, title="Relative Strength Index (RSI) from Alpha Vantage")
ax.hlines(y=30,
          xmin=rsi_df.index.min(),
          xmax=rsi_df.index.max(),
          colors="red")
ax.hlines(y=70,
          xmin=rsi_df.index.min(),
          xmax=rsi_df.index.max(),
          colors="red")

NameError: name 'plt' is not defined

: 

### 5. Khám phá đối tượng siêu dữ liệu:

In [ ]:
rsi_meta

## Tải MACD bằng API của Intrinio

### 1. Nhập thư viện:

In [ ]:
import intrinio_sdk as intrinio
import pandas as pd

### 2. Xác thực bằng khóa API cá nhân và Chọn API:

In [ ]:
intrinio.ApiClient().set_api_key("demo")
security_api = intrinio.SecurityApi()

### 3. Yêu cầu chỉ báo MACD cho cổ phiếu IBM từ năm 2020:

In [ ]:
r = security_api.get_security_price_technicals_macd(
    identifier="IBM",
    fast_period=12,
    slow_period=26,
    signal_period=9,
    price_key="close",
    start_date="2020-01-01",
    end_date="2020-12-31",
    page_size=500
)

### 4. Chuyển đổi kết quả yêu cầu thành DataFrame của pandas:

In [ ]:
macd_df = (
    pd.DataFrame(r.technicals_dict).sort_values("date_time").set_index("date_time")
)
macd_df.index = pd.to_datetime(macd_df.index).date

### 5. Vẽ biểu đồ MACD:

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

(
    macd_df[["macd_line", "signal_line"]].plot(ax=ax[0], title="Moving Average Convergence Divergence (MACD) from Intrinio")
)

ax[1].bar(df.index, macd_df["macd_histogram".values], label="macd_hist")
ax[1].legend()

## Xác định mô hình ba đường thẳng trong biểu đồ nến Bitcoin theo giờ:

### 1. Nhập các thư viện:

In [ ]:
import pandas as pd 
import yfinance as yf
import talib
import mplfinance as mpf

### 2. Tải xuống giá Bitcoin theo giờ trong 9 tháng qua:

In [ ]:
df = yf.download("BTC-USD",
                 period="9mo",
                 interval="1h",
                 progress=False)

### 3. Xác định mô hình ba đường thẳng:

In [ ]:
df["3_line_strike"] = talib.CDL3LINESTRIKE(
    df["Open"], df["High"], df["Low"], df["Close"]
)

### 4. Xác định vị trí và vẽ mô hình giảm giá:

In [ ]:
df[df["3_line_strike"] == -100].head()

In [ ]:
mpf.plot(df["2021-07-16 05:00:00":"2021-07-16 16:00:00"], type="candle")

###  5. Xác định và vẽ biểu đồ mô hình tăng giá:

In [ ]:
df[df["3_line_strike"] == 100]

In [ ]:
mpf.plot(df["2021-07-16 05:00:00":"2021-07-16 16:00:00"], type="candle")

## Xác định tất cả các mô hình có thể cùng lúc

### 1. Lấy tất cả các tên mô hình có sẵn:

In [ ]:
candle_names = talib.get_function_groups()["Pattern Recognition"]

### 2. Lặp lại danh sách các mẫu và cố gắng xác định chúng:

In [ ]:
for candle in candle_names:
    df[candle] = getattr(talib, candle)(
        df["Open"], df["High"], df["Low"], df["Close"]
    )

### 3. Kiểm tra số liệu thống kê tóm tắt của các mẫu:

In [ ]:
with pd.option_context("display.max_rows", len(candle_names)):
    display(df[candle_names].describe().transpose().round(2))